# 把 EEG 预处理做对 · BCI IV 2a 实测（A01T）

在 **BCI Competition IV Dataset 2a** 的 `A01T.gdf` 上跑一套可复现的 MNE 预处理管线：
从原始 GDF → 纯净 epoch，并给出第 7 节「同数据 · 带通± / 陷波± / ICA± / 重参考方式」的
**真实准确率与信噪比差异**。

运行本 notebook 只需把 `A01T.gdf` 放在同一目录（或改下面 `DATA` 的绝对路径），
然后 **Run All**。依赖见 `requirements.txt`。


In [ ]:
"""BCI Competition IV Dataset 2a (subject A01) - MNE preprocessing pipeline.

Provides a parameterised pipeline (re-reference / notch / bandpass / ICA) that
turns a raw GDF file into clean epochs, plus a CSP+LDA classifier that reports
classification accuracy, plus a CSP-based discriminative SNR proxy.

Author: 实验员 (MNE preprocessing do-it-right demo)
"""
import numpy as np
from collections import Counter

import mne
from mne.preprocessing import ICA
from mne.preprocessing import compute_current_source_density

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from mne.decoding import CSP

import scipy.linalg

# Canonical BCI IV 2a montage order (22 EEG channels). Verified against the
# labelled channels stored in the GDF: Fz@0, C3@7, Cz@9, C4@11, Pz@19.
CANON = ['Fz', 'FC3', 'FC1', 'FCz', 'FC2', 'FC4', 'C5', 'C3', 'C1', 'Cz',
         'C2', 'C4', 'C6', 'CP3', 'CP1', 'CPz', 'CP2', 'CP4', 'P1', 'Pz',
         'P2', 'POz']
EOG_CH = ['EOG-left', 'EOG-central', 'EOG-right']

# Event codes (decimal) from desc_2a.pdf, Table 2.
CUE_EVENTS = {1: 769, 2: 770, 3: 771, 4: 772}   # class -> raw GDF event code
CUE_EVENT_IDS = {str(v): k for k, v in CUE_EVENTS.items()}


def load_raw(gdf_path):
    """Load a BCI IV 2a GDF file, rename the 22 EEG channels to the canonical
    montage and attach the standard 10-20 positions (needed for Laplacian /
    bipolar referencing and for the montage figure)."""
    raw = mne.io.read_raw_gdf(gdf_path, preload=True)
    raw.rename_channels({raw.ch_names[i]: CANON[i] for i in range(22)})
    # mark the 3 monopolar EOG channels so that picks='eeg' selects only EEG
    raw.set_channel_types({c: 'eog' for c in EOG_CH})
    raw.set_montage(mne.channels.make_standard_montage('standard_1020'),
                    on_missing='ignore')
    return raw


def get_cue_events(raw):
    """Return events for the 4 motor-imagery cues with event ids = class 1..4,
    and the positions of the cue events belonging to expert-rejected trials
    (GDF event 1023 sits on the trial-start marker; the cue is 2 s = 500 samples
    later, per desc_2a.pdf 'cue at t=2 s')."""
    events, ev_id = mne.events_from_annotations(raw, event_id=CUE_EVENT_IDS,
                                                verbose=False)
    rej_events, _ = mne.events_from_annotations(
        raw, event_id={'1023': 1}, verbose=False)
    rejected_cue_positions = rej_events[:, 0] + 500 if len(rej_events) else \
        np.array([], int)
    return events, rejected_cue_positions


def apply_reference(raw, reference='CAR'):
    """Apply a re-referencing scheme. 'mastoid' keeps the recording reference
    (left mastoid, i.e. no offline re-reference)."""
    picks_eeg = mne.pick_types(raw.info, eeg=True)
    if reference == 'CAR':
        raw.set_eeg_reference('average', projection=False)
    elif reference == 'mastoid':
        pass  # as recorded
    elif reference == 'laplacian':
        # Spherical-spline surface Laplacian (Perrin/Cohen/Kayser-Tenke).
        # Output channels are re-labelled 'csd' with units V/m^2; the rest of the
        # pipeline selects channels by name (CANON), so the type does not matter.
        raw = compute_current_source_density(raw, sphere='auto')
    else:
        raise ValueError(f'unknown reference: {reference}')
    return raw


def apply_notch(raw, notch_freq):
    if notch_freq:
        raw.notch_filter(freqs=[notch_freq], picks=CANON,
                         notch_widths=1.0, verbose=False)
    return raw


def apply_bandpass(raw, bandpass):
    if bandpass:
        raw.filter(l_freq=bandpass[0], h_freq=bandpass[1], picks=CANON,
                   fir_design='firwin', verbose=False)
    return raw


def apply_ica(raw, eog_channels=EOG_CH, n_components=None, threshold=3.0,
              random_state=42):
    """Fit ICA on the EEG channels and remove components that correlate with
    the EOG channels (blink / eye-movement artifacts). Returns (raw, n_excluded)."""
    ica = ICA(n_components=n_components, method='fastica',
              random_state=random_state, max_iter=500)
    ica.fit(raw, picks=CANON)
    eog_bads = ica.find_bads_eog(raw, ch_name=eog_channels,
                                 threshold=threshold)[0]
    ica.exclude = sorted(set(eog_bads))
    raw = ica.apply(raw)
    return raw, len(ica.exclude)


def build_epochs(raw, events, tmin=-1.5, tmax=4.5, baseline=(-1.5, 0)):
    """Epoch the (already re-referenced/filtered/ICA'd) raw around cue events,
    retaining only the 22 EEG channels and baseline-correcting."""
    epochs = mne.Epochs(raw, events, event_id={str(k): k for k in (1, 2, 3, 4)},
                        tmin=tmin, tmax=tmax, baseline=baseline,
                        picks=CANON, preload=True, on_missing='ignore',
                        reject=None)
    return epochs


def process(raw, events, reference='CAR', notch=None, bandpass=(8, 30),
            ica=False, ica_eog_channels=EOG_CH, ica_threshold=2.5, highpass=1.0):
    """Run the full preprocessing pipeline and return (epochs, n_ica_excluded).

    Recommended order (documented in the article):
        1. re-reference
        2. notch (optional)
        3. high-pass (drift removal; also the pre-ICA band)
        4. ICA on the wideband data (EOG-correlated components removed)
        5. task band-pass (optional; *none* keeps the wideband [highpass,100 Hz])
        6. epoch
    ICA is applied *before* the narrow band-pass precisely because blink / eye
    movement artefact is low-frequency and would already be removed by a mu-band
    filter, leaving no EOG signal for the ICA to find.
    """
    raw = raw.copy()
    raw = apply_reference(raw, reference)
    raw = apply_notch(raw, notch)
    raw.filter(l_freq=highpass, h_freq=None, picks=CANON,
               fir_design='firwin', verbose=False)          # high-pass (drift)
    n_excl = 0
    if ica:
        raw, n_excl = apply_ica(raw, eog_channels=ica_eog_channels,
                                threshold=ica_threshold)
    if bandpass:
        raw = apply_bandpass(raw, bandpass)
    epochs = build_epochs(raw, events)
    return epochs, n_excl


def _drop_expert_rejected(epochs, rejected_cue_positions, tol_samples=50):
    """Drop epochs whose cue onset is within tol_samples of an expert-rejected
    trial's cue position, matching the competition's 'artifact-free trials' rule."""
    if len(rejected_cue_positions) == 0:
        return epochs
    rec = epochs.events[:, 0]
    drop = np.zeros(len(rec), bool)
    for rt in rejected_cue_positions:
        drop |= np.abs(rec - rt) <= tol_samples
    keep = ~drop
    return epochs[keep]


def classify_accuracy(epochs, tmin=0.5, tmax=4.0, n_per_class=4,
                      n_splits=5, random_state=42):
    """4-class CSP + LDA classification accuracy via stratified k-fold CV.

    Uses MultiCSP (robust one-vs-rest CSP with shrunken covariances) followed
    by an LDA classifier. Filters are fit inside each CV fold (no leakage).
    """
    X = epochs.copy().crop(tmin=tmin, tmax=tmax).get_data()
    y = epochs.events[:, 2]
    clf = make_pipeline(MultiCSP(n_per_class=n_per_class), LinearDiscriminantAnalysis())
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    scores = cross_val_score(clf, X, y, cv=cv, scoring='accuracy')
    return float(scores.mean()), float(scores.std())


class MultiCSP(BaseEstimator, TransformerMixin):
    """Robust one-vs-rest Common Spatial Patterns (multiclass) with shrunken
    covariance matrices. Each class's filters maximise that class's spatial
    variance relative to the pooled variance of the other classes; log-variance
    features are concatenated across classes. Fit on train folds only."""

    def __init__(self, n_per_class=4, alpha=0.01):
        self.n_per_class = n_per_class
        self.alpha = alpha

    def _shrink(self, cov):
        d = cov.shape[0]
        floor = self.alpha * np.trace(cov) / d
        return cov + floor * np.eye(d)

    def _spatial_cov(self, X, mask):
        Xc = X[mask]
        covs = np.einsum('ect,edt->ecd', Xc, Xc) / Xc.shape[-1]
        return covs.mean(axis=0)

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.filters_ = {}
        for c in self.classes_:
            S_c = self._spatial_cov(X, y == c)
            S_rest = self._spatial_cov(X, y != c)
            S_c = self._shrink(S_c)
            S_rest = self._shrink(S_rest)
            lam, W = scipy.linalg.eigh(S_c, S_rest)
            order = np.argsort(lam)[::-1][:self.n_per_class]
            self.filters_[c] = W[:, order]
        return self

    def transform(self, X):
        feats = []
        for c in self.classes_:
            W = self.filters_[c]
            proj = np.einsum('kc,ect->ekt', W.T, X)      # (n_ep, k, t)
            var = proj.var(axis=-1)                      # (n_ep, k)
            feats.append(np.log(var + 1e-12))
        return np.hstack(feats)


def csp_snr(epochs, tmin=0.5, tmax=4.0, alpha=0.01):
    """CSP-class-discriminative SNR proxy (dB).

    For each class the one-vs-rest generalised eigenvalue problem
        S_c w = lam * S_rest w
    is solved on the regularised spatial covariances of the imagery window; the
    largest eigenvalue lam_max is the strongest class-vs-rest power ratio.
    Reported value = 10*log10(mean(lam_max over classes)). Larger = more
    class-discriminative power = better effective signal-to-noise.
    """
    X = epochs.copy().crop(tmin=tmin, tmax=tmax).get_data()   # (n_ep, ch, t)
    y = epochs.events[:, 2]
    classes = np.unique(y)

    def spatial_cov(mask):
        Xc = X[mask]
        covs = np.einsum('ect,edt->ecd', Xc, Xc) / Xc.shape[-1]
        return covs.mean(axis=0)

    def shrink(cov):
        d = cov.shape[0]
        return cov + (alpha * np.trace(cov) / d) * np.eye(d)

    lam_max = []
    for c in classes:
        S_c = spatial_cov(y == c)
        S_rest = spatial_cov(y != c)
        S_c = shrink(S_c)
        S_rest = shrink(S_rest)
        lam = np.real(scipy.linalg.eigvalsh(S_c, S_rest))
        lam = lam[np.isfinite(lam) & (lam > 1e-12)]
        lam_max.append(lam.max())
    return float(10 * np.log10(np.mean(lam_max)))


In [ ]:
# ===================== 1. 加载数据 =====================
import os
DATA = 'A01T.gdf'                       # 同目录文件，或改成你的绝对路径
if not os.path.exists(DATA):
    for _c in ('data/A01T.gdf', '../data/A01T.gdf'):
        if os.path.exists(_c):
            DATA = _c
            break
raw = load_raw(DATA)
events, rejected_cue_pos = get_cue_events(raw)
print(f'通道数 {len(raw.ch_names)} | 提示事件 {len(events)} '
      f'(每类 {np.bincount(events[:,2])[1:]} 个) | 专家标记伪迹 trial {len(rejected_cue_pos)} 个将被剔除')

# ---- 体检（第 1 节）：采样率 / 通道 / 单位 / 事件 ----
print('采样率', raw.info['sfreq'], 'Hz | 时长 %.1f s' % raw.times[-1])
print('前 22 个为 EEG，最后 3 个为 EOG（不参与分类）')


In [ ]:
# ===================== 2. 预处理 =====================
# 一次跑一条流水线：重参考 -> 陷波 -> 高通(去漂移/ICA前置) -> ICA -> 任务带通 -> 分段
def run_config(reference, notch, bandpass, ica, highpass=1.0, ica_threshold=2.5):
    ep, n_excl = process(raw, events, reference=reference, notch=notch,
                         bandpass=bandpass, ica=ica, highpass=highpass,
                         ica_threshold=ica_threshold)
    ep = _drop_expert_rejected(ep, rejected_cue_pos)
    return ep, n_excl


In [ ]:
# ===================== 3. 第 7 节对比网格 =====================
import pandas as pd
CONFIGS = [
    ('C1  mastoid  | 宽带    | 无陷波 | 无ICA', 'mastoid',  None, None,        False),
    ('C2  CAR      | 宽带    | 无陷波 | 无ICA', 'CAR',      None, None,        False),
    ('C3  CAR      | 宽带    | 50Hz   | 无ICA', 'CAR',      50,   None,        False),
    ('C4  CAR      | 宽带    | 50Hz   | ICA',   'CAR',      50,   None,        True),
    ('C5  CAR      | 8-30Hz  | 无陷波 | ICA  [推荐]', 'CAR', None, (8, 30),   True),
    ('C6  CAR      | 8-30Hz  | 无陷波 | 无ICA', 'CAR',      None, (8, 30),    False),
    ('C7  mastoid  | 8-30Hz  | 无陷波 | ICA',   'mastoid',  None, (8, 30),    True),
    ('C8  laplacian| 8-30Hz  | 无陷波 | ICA',   'laplacian',None, (8, 30),    True),
]

rows = []
for label, ref, notch, band, ica in CONFIGS:
    ep, n_excl = run_config(ref, notch, band, ica)
    acc, acc_std = classify_accuracy(ep)
    snr = csp_snr(ep)
    rows.append({'config': label, 'reference': ref,
                 'bandpass': f'{band}' if band else 'wideband(1-100Hz)',
                 'notch': 'no' if not notch else f'{notch}Hz',
                 'ica': 'yes' if ica else 'no', 'ica_excluded': n_excl,
                 'n_epochs': len(ep.events), 'accuracy': round(acc, 4),
                 'accuracy_std': round(acc_std, 4), 'snr_db': round(snr, 3)})
    print(f'{label:38s} acc={acc:.3f}±{acc_std:.3f}  snr={snr:.2f}dB  '
          f'n={len(ep.events)}  ica_ex={n_excl}')

df = pd.DataFrame(rows)
df.to_csv('comparison_table.csv', index=False)
print('\n第 7 节实测对比表（A01T，273 个无伪迹 trial，5 折 CV，CSP+LDA）：')
print(df.to_string(index=False))


In [ ]:
# ===================== 4. 信噪比 / 频谱佐证图 =====================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def band_p(fre, P, f0, f1):
    m = (fre >= f0) & (fre < f1); return P[m].mean()

# 顶图：频谱（说明 CAR / 陷波 / 带通各做了什么）
raw_eeg = raw.copy().pick(CANON)
psd = raw_eeg.compute_psd(method='welch', fmin=1, fmax=120, n_fft=4096, verbose=False)
fre, Praw = psd.freqs, psd.get_data().mean(axis=0) * 1e12

rc = raw.copy(); rc.set_eeg_reference('average', projection=False); rc = rc.pick(CANON)
psd = rc.compute_psd(method='welch', fmin=1, fmax=120, n_fft=4096, verbose=False)
Pcar = psd.get_data().mean(axis=0) * 1e12

rb = raw.copy(); rb.set_eeg_reference('average', projection=False)
rb.filter(l_freq=8, h_freq=30, picks=CANON, fir_design='firwin', verbose=False)
rb = rb.pick(CANON)
psd = rb.compute_psd(method='welch', fmin=1, fmax=120, n_fft=4096, verbose=False)
Pbp = psd.get_data().mean(axis=0) * 1e12

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(fre, Praw, lw=1.4, label='raw (mastoid ref)')
ax.semilogy(fre, Pcar, lw=1.4, label='CAR')
ax.semilogy(fre, Pbp, lw=1.4, label='CAR + 8-30Hz')
ax.set_xlim(0, 120); ax.set_xlabel('Hz'); ax.set_ylabel('µV²/Hz (log)')
ax.legend(fontsize=8); ax.grid(alpha=.3)
ax.set_title('A01T 频谱：重参考 / 带通的作用')
plt.savefig('spectrum_preprocessing.png', dpi=150); plt.show()

# 佐证「陷波未生效」：原始数据 50Hz 已是陷波后（峰/邻比<1 即无 50Hz 线噪峰）
c = band_p(fre, Praw, 49.5, 50.5); nb = (band_p(fre, Praw, 47, 49) + band_p(fre, Praw, 51, 53)) / 2
print(f'\n50Hz 峰/邻比（原始数据）= {c/nb:.3f}  (<1 ⇒ 放大器已陷波，无残留线噪峰可切)')
print('mu8-13: raw=%.2f  CAR=%.3f  CAR+8-30=%.3f µV²/Hz' % (
    band_p(fre, Praw, 8, 13), band_p(fre, Pcar, 8, 13), band_p(fre, Pbp, 8, 13)))


## 在线性能 / GPU / 降采样实测（正文 §10 / §11 / §12 数字来源 · W-131）

本 notebook 只跑第 7 节离线管线（C1–C8 对比）。正文新增的 **§10 降采样抗混叠 / §11 在线性能 / §12 GPU 加速** 的「本数据实测」数字，由另一实验归档提供：

- 路径：`E:\CherryClaw\projects\bci-content-lab\在线性能与GPU加速实测\`
- 脚本：`scripts/{measure_timing, measure_phase, measure_rtf, measure_projection, measure_eegnet, measure_downsample, make_figures}.py`
- 数字与图：`results/README_在线性能_GPU实测.md`、`results/*.json`、`cpu_vs_gpu_timing.png`、`phase_delay_demo.png`、`downsample_antialias.png`

环境：CPU i5-12490F ｜ GPU RTX 3060 8GB（torch 2.6.0 + cu124）；数据 A01T / 22 导 / 250 Hz / 273 clean MI trial。
